# Marketing Ads Analysis
Reproduces the full Marketing - Ads tab from the NBS BI dashboard.
Edit the **Parameters** cell to change the analysis window, platform filter, or referral code.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from nbs_bi.config import ADS_DATABASE_URL, READONLY_DATABASE_URL
from nbs_bi.clients.campaigns import (
    CampaignAnalyzer,
    load_ad_spend_from_db,
    aggregate_spend,
)
from nbs_bi.clients.report import ClientReport
from nbs_bi.onramp.queries import OnrampQueries
from nbs_bi.reporting.cards import _load_all_invoice_models
from nbs_bi.reporting.marketing import (
    _build_cumulative_spend,
    _build_channel_comparison,
    _fig_cumulative_spend,
    _fig_cumulative_profit,
    _fig_revenue_breakdown,
    _fig_campaign_roi,
    _fig_campaign_cac,
    _fig_campaign_daily,
    _fig_daily_revenue_vs_spend,
    _fig_daily_rev_all_vs_cohort,
    _fig_channel_comparison,
    _fig_channel_daily,
    _fig_campaign_funnel,
)

print('Imports OK')

2026-04-30 18:54:24.617 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-30 18:54:24.618 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-04-30 18:54:24.618 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Imports OK


## Parameters

In [2]:
# --- Edit these ---
START_DATE    = '2025-06-23'   # earliest spend date; set to None for all history
END_DATE      = str(pd.Timestamp.today().date())
PLATFORM      = None           # None = all, 'meta', 'google'
REFERRAL_CODE = ''             # '' = no filter; e.g. 'ERIC'

print(f'Analysis window : {START_DATE} → {END_DATE}')
print(f'Platform filter : {PLATFORM or "all"}')
print(f'Referral code   : {REFERRAL_CODE or "none"}')

Analysis window : 2025-06-23 → 2026-04-30
Platform filter : all
Referral code   : none


## 1. Ad Spend Data

In [3]:
spend_df_raw = load_ad_spend_from_db(ADS_DATABASE_URL)

# Apply date window
if START_DATE:
    spend_df_raw = spend_df_raw[pd.to_datetime(spend_df_raw['date']) >= pd.Timestamp(START_DATE)]
if END_DATE:
    spend_df_raw = spend_df_raw[pd.to_datetime(spend_df_raw['date']) <= pd.Timestamp(END_DATE)]

# Platform filter
spend_df = spend_df_raw.copy()
if PLATFORM:
    spend_df = spend_df[spend_df['platform'] == PLATFORM]

spend_agg = aggregate_spend(spend_df)

print(f'Rows loaded : {len(spend_df_raw)}')
print(f'Date range  : {pd.to_datetime(spend_df_raw["date"]).min().date()} → {pd.to_datetime(spend_df_raw["date"]).max().date()}')
print(f'Platforms   : {sorted(spend_df_raw["platform"].unique().tolist())}')
print(f'Total spend : ${spend_df_raw["daily_spend_usd"].sum():,.2f}')
spend_df_raw.tail()

Rows loaded : 37
Date range  : 2025-06-23 → 2026-04-30
Platforms   : ['google', 'meta']
Total spend : $9,277.96


,date,platform,daily_spend_usd
32,2026-04-28,google,222.07
33,2026-04-28,meta,395.00
34,2026-04-29,google,248.61
35,2026-04-29,meta,393.00
36,2026-04-30,meta,131.00


## 2. Campaign Detection

In [4]:
analyzer  = CampaignAnalyzer(spend_agg, db_url=READONLY_DATABASE_URL)
campaigns = analyzer.campaigns

print(f'Detected {len(campaigns)} campaign(s):')
for c in campaigns:
    print(f"  {c['campaign_id']}  {c['start']} → {c['end']}  ${c['total_spend_usd']:,.2f}")

Detected 10 campaign(s):
  campaign_1  2025-06-23 → 2025-06-23  $34.48
  campaign_2  2025-07-28 → 2025-07-28  $258.62
  campaign_3  2025-08-27 → 2025-08-27  $273.45
  campaign_4  2025-09-23 → 2025-09-23  $280.44
  campaign_5  2025-10-27 → 2025-10-27  $557.49
  campaign_6  2025-12-17 → 2025-12-17  $903.36
  campaign_7  2026-01-26 → 2026-02-02  $1,320.67
  campaign_8  2026-02-15 → 2026-02-16  $890.93
  campaign_9  2026-02-26 → 2026-03-04  $501.21
  campaign_10  2026-04-14 → 2026-04-30  $4,257.30


## 3. ROI Summary

In [5]:
summary = analyzer.roi_summary()
summary

,campaign_id,start,end,duration_days,total_spend_usd,cohort_users,transacting_users,transacting_rate,baseline_rate_per_day,incremental_users_est,total_revenue_usd,roas,cac_full,cac_incremental,avg_rev_per_transacting_user
0,campaign_1,2025-06-23,2025-06-23,1,34.48,0,0,NaN,0.0,0.0,0.00,0.0000,NaN,NaN,NaN
1,campaign_2,2025-07-28,2025-07-28,1,258.62,0,0,NaN,0.0,0.0,0.00,0.0000,NaN,NaN,NaN
2,campaign_3,2025-08-27,2025-08-27,1,273.45,16,8,0.5000,13.9,2.0,76.15,0.2785,34.18,127.61,9.52
3,campaign_4,2025-09-23,2025-09-23,1,280.44,10,1,0.1000,20.7,0.0,0.00,0.0000,280.44,NaN,0.00
4,campaign_5,2025-10-27,2025-10-27,1,557.49,75,14,0.1867,53.0,22.0,142.26,0.2552,39.82,25.34,10.16
5,campaign_6,2025-12-17,2025-12-17,1,903.36,10,1,0.1000,22.1,0.0,45.56,0.0504,903.36,NaN,45.56
6,campaign_7,2026-01-26,2026-02-02,8,1320.67,842,139,0.1651,88.0,138.0,1821.52,1.3792,9.50,9.57,13.10
7,campaign_8,2026-02-15,2026-02-16,2,890.93,58,8,0.1379,69.4,0.0,93.80,0.1053,111.37,NaN,11.73
8,campaign_9,2026-02-26,2026-03-04,7,501.21,276,54,0.1957,57.7,0.0,1439.75,2.8725,9.28,NaN,26.66
9,campaign_10,2026-04-14,2026-04-30,17,4257.30,2166,94,0.0434,15.3,1906.0,2180.89,0.5123,45.29,2.23,23.20


## 4. KPIs

In [6]:
latest_id     = summary['campaign_id'].iloc[-1]
kyc_done      = analyzer.cohort_kyc_count(latest_id, referral_code=REFERRAL_CODE)
total_spend   = summary['total_spend_usd'].sum()
total_revenue = summary['total_revenue_usd'].sum()
transacting   = int(summary['transacting_users'].sum())
cohort_users  = int(summary['cohort_users'].sum())
roas = total_revenue / total_spend if total_spend else float('nan')

print(f'Latest campaign   : {latest_id}')
print(f'Total ad spend    : ${total_spend:,.2f}')
print(f'Cohort revenue    : ${total_revenue:,.2f}')
print(f'Overall ROAS      : {roas:.2f}×')
print(f'Cohort sign-ups   : {cohort_users:,}')
print(f'KYC completed     : {kyc_done:,}')
print(f'Activated users   : {transacting:,}')

Latest campaign   : campaign_10
Total ad spend    : $9,277.95
Cohort revenue    : $5,799.93
Overall ROAS      : 0.63×
Cohort sign-ups   : 3,453
KYC completed     : 600
Activated users   : 319


## 5. Cumulative Revenue & Profit

In [7]:
cum_rev_df = analyzer.cumulative_revenue(latest_id, referral_code=REFERRAL_CODE)

_, _, _, history = _load_all_invoice_models()
invoice_history = [
    (period, m.inputs.invoice_total_usd or float(m.cost_breakdown().total), m.inputs.n_transactions)
    for period, m in history
]

cum_profit_df = analyzer.cumulative_profit(
    latest_id, invoice_history, referral_code=REFERRAL_CODE
)

print(f'Cohort revenue dataframe : {len(cum_rev_df)} rows')
print(f'Cumulative revenue       : ${cum_rev_df["cum_rev_usd"].iloc[-1]:,.2f}')
cum_profit_df[[
    'date', 'cum_rev_usd', 'cum_card_cogs_usd',
    'cum_contribution_margin_usd', 'cum_profit_usd'
]].tail()

2026-04-30 18:54:33.424 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-04-30 18:54:33,424 | nbs_bi.cards.invoice_parser | INFO | Loaded invoice inputs from /home/david/Documents/nbs/repos/nbs_bi/data/invoices/Invoice-NKEMEJLO-0008-actuals.json


2026-04-30 18:54:33,425 | nbs_bi.cards.invoice_parser | INFO | Loaded invoice inputs from /home/david/Documents/nbs/repos/nbs_bi/data/invoices/Invoice-NKEMEJLO-0009-actuals.json


Cohort revenue dataframe : 17 rows
Cumulative revenue       : $2,180.89


,date,cum_rev_usd,cum_card_cogs_usd,cum_contribution_margin_usd,cum_profit_usd
12,2026-04-26,1084.0654,46.087754,-1484.642354,1037.977646
13,2026-04-27,1497.4934,56.204578,-1426.331178,1441.288822
14,2026-04-28,1788.0257,68.569585,-1765.233885,1719.456115
15,2026-04-29,1986.3087,107.912790,-2247.904090,1878.395910
16,2026-04-30,2180.8906,133.766896,-2210.176296,2047.123704


## 6. Daily Context

In [8]:
daily = analyzer.daily_context()
daily.tail(10)

,new_signups,date,daily_spend_usd,is_campaign,campaign_id
252,124,2026-04-21,149.00,True,campaign_10
253,594,2026-04-22,473.63,True,campaign_10
254,317,2026-04-23,295.15,True,campaign_10
255,151,2026-04-24,354.84,True,campaign_10
256,216,2026-04-25,301.00,True,campaign_10
257,161,2026-04-26,216.00,True,campaign_10
258,74,2026-04-27,345.00,True,campaign_10
259,105,2026-04-28,617.07,True,campaign_10
260,135,2026-04-29,641.61,True,campaign_10
261,111,2026-04-30,131.00,True,campaign_10


## 7. Acquisition & Channel Data

In [9]:
client_report = ClientReport(
    START_DATE or '2025-06-01', END_DATE, db_url=READONLY_DATABASE_URL
).build()
acquisition            = client_report.get('acquisition')
profit_by_source_daily = client_report.get('profit_by_source_daily')
print('Acquisition sources:', sorted(acquisition['acquisition_source'].unique().tolist()) if acquisition is not None else 'n/a')

2026-04-30 18:54:34,474 | nbs_bi.clients.queries | INFO | DB query [cohort_base] None → None


2026-04-30 18:54:34,902 | nbs_bi.clients.queries | INFO | DB query [conv_rev] 2025-06-23 → 2026-05-01


2026-04-30 18:54:35,029 | nbs_bi.clients.queries | INFO | DB query [card_fees] None → None


2026-04-30 18:54:35,093 | nbs_bi.clients.queries | INFO | DB query [card_txs] 2025-06-23 → 2026-05-01


2026-04-30 18:54:35,138 | nbs_bi.clients.queries | INFO | DB query [billing] 2025-06-23 → 2026-05-01


...

2026-04-30 18:54:38,657 | nbs_bi.clients.queries | INFO | DB query [card_txs_monthly] 2025-06-23 → 2026-05-01


2026-04-30 18:54:39,245 | nbs_bi.clients.queries | INFO | DB query [daily_activity] 2025-06-23 → 2026-05-01


2026-04-30 18:54:39,664 | nbs_bi.clients.queries | INFO | DB query [activity_kpis] None → None


2026-04-30 18:54:39,713 | nbs_bi.clients.queries | INFO | DB query [signups_24h] None → None


Acquisition sources: ['direct_referral', 'founder_invite', 'organic']


## 8. All-Users Daily Revenue (platform baseline)

In [10]:
date_start = str(pd.to_datetime(cum_rev_df['date']).min().date())
date_end   = str((pd.Timestamp.today() + pd.Timedelta(days=1)).date())

all_users_rev_df = OnrampQueries(
    start_date=date_start, end_date=date_end, db_url=READONLY_DATABASE_URL
).daily_revenue_by_product()

print(f'All-users daily revenue : {len(all_users_rev_df)} rows')
all_users_rev_df.tail()

2026-04-30 18:54:39,809 | nbs_bi.onramp.queries | INFO | DB query [conversions] 2026-04-14 → 2026-05-02


2026-04-30 18:54:40,107 | nbs_bi.onramp.queries | INFO | DB query [card_fees_daily] 2026-04-14 → 2026-05-02


2026-04-30 18:54:40,154 | nbs_bi.onramp.queries | INFO | DB query [billing_daily] 2026-04-14 → 2026-05-02


2026-04-30 18:54:40,205 | nbs_bi.onramp.queries | INFO | DB query [swaps_daily] 2026-04-14 → 2026-05-02


All-users daily revenue : 17 rows


,date,daily_rev_conversion_usd,daily_rev_card_fees_usd,daily_rev_billing_usd,daily_rev_swap_usd,daily_rev_usd
12,2026-04-26,75.004504,350.0,201.806798,14.663354,641.474657
13,2026-04-27,80.387381,400.0,163.713542,6.696595,650.797518
14,2026-04-28,218.833390,250.0,170.816797,2.526615,642.176801
15,2026-04-29,284.945624,50.0,306.864036,11.371505,653.181165
16,2026-04-30,216.151471,150.0,192.138764,12.658949,570.949184


---
## Charts

### Cohort Activation Funnel

In [11]:
funnel = {'signups': cohort_users, 'kyc_done': kyc_done, 'activated': transacting}
fig = _fig_campaign_funnel(funnel)
if fig:
    fig.show()

### Cumulative Spend vs Cohort Revenue

In [12]:
cum_df = _build_cumulative_spend(spend_agg, campaigns)
fig = _fig_cumulative_spend(cum_df, campaigns, cum_rev_df, cum_profit_df)
if fig:
    fig.show()

### Operational Profit & Contribution Margin

In [13]:
fig = _fig_cumulative_profit(cum_profit_df)
if fig:
    fig.show()

### Revenue Breakdown (stacked area)

In [14]:
fig = _fig_revenue_breakdown(cum_profit_df)
if fig:
    fig.show()

### Ad Spend vs Cohort Revenue by Campaign

In [15]:
fig = _fig_campaign_roi(summary, cum_profit_df)
if fig:
    fig.show()

### Customer Acquisition Cost (CAC)

In [16]:
fig = _fig_campaign_cac(summary)
if fig:
    fig.show()

### Daily Sign-ups vs Ad Spend

In [17]:
fig = _fig_campaign_daily(daily)
if fig:
    fig.show()

### Daily Revenue vs Ad Spend

In [18]:
fig = _fig_daily_revenue_vs_spend(cum_rev_df, spend_agg)
if fig:
    fig.show()

### Platform Revenue vs Cohort Boundary

In [19]:
fig = _fig_daily_rev_all_vs_cohort(all_users_rev_df, cum_rev_df, spend_agg)
if fig:
    fig.show()

### Channel Comparison

In [20]:
comparison = _build_channel_comparison(summary, acquisition, cum_profit_df)

fig = _fig_channel_comparison(comparison)
if fig:
    fig.show()

if profit_by_source_daily is not None:
    fig2 = _fig_channel_daily(profit_by_source_daily)
    if fig2:
        fig2.show()

comparison

,acquisition_source,n_users,avg_operational_profit_usd,total_operational_profit_usd,conversion_rate,spend_usd,roas,n_transacting,median_net_revenue_usd
0,meta_ads,3453,0.592854,2047.123704,0.092383,9277.95,0.220644,NaN,NaN
1,direct_referral,7452,0.868737,6473.828451,0.099436,NaN,NaN,741.0,-2.07
2,founder_invite,2019,2.821600,5696.809434,0.147103,NaN,NaN,297.0,-2.07
3,organic,3995,0.021713,86.745170,0.065582,NaN,NaN,262.0,-2.07


---
## Campaign Summary Table

In [21]:
display(summary)

,campaign_id,start,end,duration_days,total_spend_usd,cohort_users,transacting_users,transacting_rate,baseline_rate_per_day,incremental_users_est,total_revenue_usd,roas,cac_full,cac_incremental,avg_rev_per_transacting_user
0,campaign_1,2025-06-23,2025-06-23,1,34.48,0,0,NaN,0.0,0.0,0.00,0.0000,NaN,NaN,NaN
1,campaign_2,2025-07-28,2025-07-28,1,258.62,0,0,NaN,0.0,0.0,0.00,0.0000,NaN,NaN,NaN
2,campaign_3,2025-08-27,2025-08-27,1,273.45,16,8,0.5000,13.9,2.0,76.15,0.2785,34.18,127.61,9.52
3,campaign_4,2025-09-23,2025-09-23,1,280.44,10,1,0.1000,20.7,0.0,0.00,0.0000,280.44,NaN,0.00
4,campaign_5,2025-10-27,2025-10-27,1,557.49,75,14,0.1867,53.0,22.0,142.26,0.2552,39.82,25.34,10.16
5,campaign_6,2025-12-17,2025-12-17,1,903.36,10,1,0.1000,22.1,0.0,45.56,0.0504,903.36,NaN,45.56
6,campaign_7,2026-01-26,2026-02-02,8,1320.67,842,139,0.1651,88.0,138.0,1821.52,1.3792,9.50,9.57,13.10
7,campaign_8,2026-02-15,2026-02-16,2,890.93,58,8,0.1379,69.4,0.0,93.80,0.1053,111.37,NaN,11.73
8,campaign_9,2026-02-26,2026-03-04,7,501.21,276,54,0.1957,57.7,0.0,1439.75,2.8725,9.28,NaN,26.66
9,campaign_10,2026-04-14,2026-04-30,17,4257.30,2166,94,0.0434,15.3,1906.0,2180.89,0.5123,45.29,2.23,23.20


## Referral Code Breakdown (optional)

In [22]:
# Lists all referral codes tracked in the DB for this campaign
options = analyzer.referral_code_options()
print(f'Available referral codes ({len(options)}):', options)

Available referral codes (115): ['0XCOKINHA', 'ACCOMPLISH', 'ANALUIZAFAN', 'ANDERSONROD', 'ANDREIAMARTINBR', 'ANTONI', 'BCRIPTO', 'BILLYSABIA', 'BITDASMINAS', 'BRENNER', 'BRLIMAN', 'BRUNAELEUTERIO', 'BRUNOBARBOSA', 'BRUNOCOSTA', 'BRUNOLIVEIRATDR', 'BUDATR', 'BUSINESS', 'CADASTRO-FM', 'CADUDG', 'CAMILLO', 'CARELLI', 'CAROLINVEST', 'CASSIOMMJ', 'CLAISSON', 'CRIPTOCAMPOS', 'CRIPTOFORJA', 'CRIPTOMEMEBR', 'CRYPTOGURI', 'CRYPTORICO', 'DASHI', 'DAVIDROCHA', 'DAZA', 'DIMERDOLFINI', 'DOGDAMASSA', 'DOLLAR', 'DONTIEPO', 'EXECUTIVO', 'FILIPE', 'FORIGO', 'GABRIELSOWER', 'GAUCHO', 'GOOGLE', 'GSX', 'HATAB', 'HIGOR', 'IGOR', 'ISAAC', 'ISAACARAUJO', 'JFT', 'JHON', 'JOAOLUCAS', 'JOAOPEDROBELLO', 'JONATARIBAS', 'JONPS', 'JOUBERT', 'JUANSB', 'JULIANOBONINSENHA', 'KATIANO', 'KELVIN', 'KPONYKCRYPTO', 'L0CK', 'LELOROSA', 'LEOCUBAS', 'LGND', 'LIDYANE', 'LIPESOLANA', 'LNOVELLI', 'LOMBRAJR', 'LPCONSULTORIA', 'LUCASNBS', 'MANOEL', 'MARCELO', 'MARICRIPTO', 'MELO', 'MODERNIZE', 'MOENSTER', 'MOONVILLA', 'MOREIRA', 